# A RAG-enhanced chatbot

So, in this example we will build a chatbot backend that will be providing tech support on the GNU make utility<sup>1</sup>.

---
<p><small>1. The reason for choosing the GNU make utility as the subject is two-fold: 1) the normal (and frequent) use of "make" is as a verb and has nothing to do with the noun "make" which forces the LLM to make(!) a non-trivial distinctions, and 2) the GNU Make Manual is freely available for download in the context of educational purposes like this. Unfortunately, it is also a case where RAG is somewhat unnecessary due to the abundance of make questions on the internet. Feel free to replace it with something of your own choice.</small></p>

---

## Prerequisites

The very first step is to make sure all requirements (in terms of python modules) are satisfied.

In [ ]:
!pip -q install ollama qdrant-client==1.11.3

## A Basic Chatbot

First up is a simple chatbot class that keeps a record of the N latest query/answer pairs generated. 

For this we will use the `chat()` method of the client rather than `generate()` that we used in the previous part. The main difference is that instead of a prompt, we pass it a list of messages repreresenting the conversation this far.
The messages are tagged with a _role_ depending on who said what, see this [notebook](./LLM-role-context.ipynb) for a longer explanation. 

Take a minute to appreciate the linguistic complexity of the instructions to the LLM in `sysmsg()` below.

In [ ]:
import ollama
from inspect import cleandoc

class ChatClient:
    """A basic chat client that keeps a record of the N latest query/answer pairs generated."""

    N = 3

    def __init__(self, ollama_host, ollama_model):
        self.client = ollama.Client(host=ollama_host)
        self.model = ollama_model
        self.messages = [{'role': 'system', 'content': self.system_prompt()}]

    def _post(self, query_msg):
        # drop oldest query+answer pair from history if there are more than N pairs
        if len(self.messages) > 2*self.N+1:
            self.messages = self.messages[:1]+self.messages[3:]
        # add the query to the message history
        self.messages.append(query_msg)

        response = self.client.chat(
            model=self.model,
            messages=self.messages,
            stream=False,
        )
        reply = response.message
        # add the reply to the message history
        self.messages.append(reply)
        return reply['content']

    def system_prompt(self):
        prompt = """
            You are an AI assistant providing tech support on the GNU Make program.
            In this context GNU Make, Make, and 'make' always refer to the GNU Make program,
            and so does the noun make.
            Provide short, concise answers prefixed by >>> .
            If you cannot answer the question, just say so.
            """
        return cleandoc(prompt)

    def post(self, query):
        # posting a query using a simple template, receiving (and printing) the answer
        msg = {'role': 'user', 'content': f'{query}'}
        answer = self._post(msg)
        print(answer)

A helper

In [ ]:
import json
from IPython import display

def show_messages(messages):
    """Helper function to view messages in a nice format"""
    s = json.loads(json.dumps(messages, default=dict))
    return display.JSON(s, root="Message history")

After instantiating a chatbot ...

In [ ]:
OLLAMA_HOST = 'http://10.129.20.4:9090'
OLLAMA_MODEL = 'qwen3:4b' # 'llama3:70b'

client = ChatClient(OLLAMA_HOST, OLLAMA_MODEL)

... we can pose it a series of questions. Note that there is less and less explicit context in the questions, with the last one only implicitly referring to the core question (compiling latex into pdf).

In [ ]:
client.post("What can I use make for?")

In [ ]:
client.post("Can you give a simple example of how to compile a single-file C project?")

In [ ]:
client.post("What would a similar example of compiling latex into pdf look like?")

In [ ]:
client.post("but I don't have pdflatex on my system")

Take a look at the message history so far, take note of the first user message (1)

In [ ]:
show_messages(client.messages)

In [ ]:
client.post("I only have xetex")

In [ ]:
show_messages(client.messages)

Notice how the message history no longer grows, and the oldest entry (1) no longer is "What can I use make for?"

## Preparing content for RAG

This has been explained in the previous part, and will thus be stated without much commentry.

### Chunking

Rather than just taking sentences as chunks, we'll bundle bundle a number of sentences into chunks, respecting the (approximate) maximum chunksize (`CHUNKSIZE=1200`).

In [ ]:
#
# Split the input data into chunks
#
import re

class Chunker:

    CHUNKSIZE=1200

    def __init__(self, filepath):
        with open(filepath, 'r') as fd:
            text = fd.read()
        self.sentences = re.split(r"(?:(?<=\.|\?|!)\n)|(?:\n(?=\s*\d{1,}(?:\.\d)*))|(?:\n\n)", text)

    def _chunk_sentences(self, idx):
        chunk = ""
        while len(chunk) < self.CHUNKSIZE and idx < len(self.sentences):
            chunk = chunk + "\n" + self.sentences[idx]
            idx += 1
        return chunk, idx

    def chunk_text(self):
        chunks = []
        next_idx = 0
        while next_idx < len(self.sentences):
            first_idx = next_idx
            chunk, next_idx = self._chunk_sentences(first_idx)
            # get number of sentences in chunk and MAYBE back up to get a slight overlap
            # n = next_idx - first_idx
            # print(first_idx, n)
            chunks.append(chunk)

        return chunks

In [ ]:
gmm = Chunker('gmm/gnu-make-manual.txt')
chunks = gmm.chunk_text()

Let's have a look at a few chunks:

In [ ]:
import json
from IPython import display

entries = json.loads(json.dumps(chunks[100:106], default=dict))
display.JSON(entries, root=f"Five chunks out of {len(chunks)}")

### Embedding

This is straightforward, except that embeddings are cached to save time on repeated runs.

In [ ]:
class Embedder:

    def __init__(self, embedding_model_name, embedding_host):
        # Set the model name
        self.embedding_model_name = embedding_model_name
        self.client = ollama.Client(host=embedding_host)
        self._cache = []
        self._hash = None

    def embedding_size(self):
        return len(self.embed_query("Dummy query"))
    
    def embed_query(self, query):
        response = self.client.embed(model=self.embedding_model_name, input=query)
        return response.embeddings[0]
        
    def embed(self, chunks, force_compute = False):
        # Vectorize, i.e. create embeddings
        # This can take a couple of minutes,
        # so use cached embeddings unless something changed
        _hash = hash((tuple(chunks), self.embedding_model_name))
        compute = len(self._cache) == 0 or self._hash != _hash or force_compute
        if compute:
            self._hash = _hash
            self._cache = []
            N_CHUNKS = 200
            for i in range(0, len(chunks), N_CHUNKS):
                print(f"processing chunks {i} to {i+N_CHUNKS-1}...")
                response = self.client.embed(model=self.embedding_model_name, input=chunks[i:i+N_CHUNKS])
                self._cache.extend(response.embeddings)

        return self._cache

In [ ]:
embedder = Embedder('mxbai-embed-large', OLLAMA_HOST)
embeddings = embedder.embed(chunks, force_compute=True)
print(len(embeddings))

Test caching:

In [ ]:
embeddings = embedder.embed(chunks)
print(len(embeddings))

### Vector store

**Please note that you must define a name for the collection below.**

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Distance, VectorParams

class VectorStore:

    # Provide a name for the _collection_ making up your corner of the database
    # use e.g. <signum>_gmm_date
    collection_name =  # <-- string with name here, e.g. "test-<your_name>-<date>"

    def __init__(self, host, port, embedder):
        # Create a client connecting to the service
        self.db = QdrantClient(host=host, port=port)
        self.embedder = embedder
 
    def _clear(self):
        # Check if collection (for this toy example) already exist, and remove if so
        if self.db.collection_exists(collection_name=self.collection_name):
           self.db.delete_collection(collection_name=self.collection_name)

        # Create a named collection and set vector dimension and metric (EUCLID => L2)
        self.db.create_collection(
            collection_name = self.collection_name,
            vectors_config = VectorParams(
                size = self.embedder.embedding_size(),
                distance = Distance.EUCLID
            ),
        )

    # Upload our embeddings, one variant of many (destroys old data)
    def upload(self, embeddings, payload):
        self._clear()
        db_points = [
            models.PointStruct(
                id=idx+1,
                vector=embeddings[idx],
                payload={'text': payload[idx]},
            ) 
            for idx in range(len(embeddings))]
        self.db.upsert(
            collection_name = self.collection_name,
            points = db_points
        )

    def query(self, question, n = 1):
        # Return the n closest matches
        search_results = self.db.search(
            collection_name = self.collection_name,
            search_params = models.SearchParams(hnsw_ef=10, exact=False),
            query_vector = self.embedder.embed_query(question),
            limit = n,
        )

        return [(result.id, result.payload, result.score) for result in search_results]

### Uploading to the store

In [ ]:
VECTORSTORE_HOST = "10.129.20.4"
VECTORSTORE_PORT = 6333

store = VectorStore(VECTORSTORE_HOST, VECTORSTORE_PORT, embedder)
store.upload(embeddings, chunks)

### Testing the store

In [ ]:
res = store.query("How can I use make to compile a program?", 3)

In [ ]:
for idx, payload, score in res:
    print(f"{idx}: (score={score})\n{payload}\n\n")

### Building a RAG-enabled chat client

Finally, we have to subclass the chat client to augment(!) it to accommodate the RAG context.

The question is where to put the retrieved content. Viable options are e.g.

1. With every 'user' message
2. Inject it in the 'system' message every time

The first option is wasteful in terms of the limited _context window_ that LLM's can handle particularly as the message history grows with the number of questions.
Tthe second option is technically a little tricky to get right.

Here we'll go for a simple but efficient alternative:  
Put the retrieved content in the 'user' message, but remove it from the message history once we have the response.

In [ ]:
class RAGChatClient(ChatClient):
    """A RAG chat client that keeps a record of the N latest query/answer pairs generated and provides a RAG context."""

    CTX_SIZE = 3

    def __init__(self, ollama_host, ollama_model, vectorstore_host, vectorstore_port, chunks, embedder):
        super().__init__(ollama_host, ollama_model)
        self.store = VectorStore(vectorstore_host, vectorstore_port, embedder)

    def system_prompt(self):
        prompt = super().system_prompt()
        rag_addition = """
        You may get additional facts prepended to the user's questions. 
        Consider the provided facts when you answer. State any referenced fact sections by number.
        """
        return prompt + "\n\n" + cleandoc(rag_addition)

    def post(self, query):
        # get RAG context
        res = store.query(query, self.CTX_SIZE)
        ctxs = [f"### Section {idx}\n{payload['text']}" for idx, payload, _ in res]
        facts = "\n\n".join(ctxs) + "\n\n"
        query_preamble = "## Facts\n\nHere’s knowledge automatically retrieved based on the user question.\n\n"
        # Reusing `post()` for posting a query using a template, receiving (and printing) the answer
        super().post(query_preamble + facts + "## User question\n\n" + query)
        # Clean up second to last message (remove RAG additions from message history)
        # Comment the next line to see effect of NOT purging RAG additions
        self.messages[-2] = {'role': 'user', 'content': f'{query}'}

## A RAG-enabled chat client

With that we can try out the improved chat client. Let's test it with the same sequence of questions as previously:

In [ ]:
client = RAGChatClient(OLLAMA_HOST, OLLAMA_MODEL, VECTORSTORE_HOST, VECTORSTORE_PORT, chunks, embedder)

In [ ]:
client.post("What can I use make for?")

Verify that the RAG additions have been purged from user message (1)

In [ ]:
show_messages(client.messages)

In [ ]:
client.post("Can you give a simple example of how to compile a single-file C project?")

In [ ]:
client.post("What would a similar example of compiling latex into pdf look like?")

In [ ]:
client.post("but I don't have pdflatex on my system")

In [ ]:
client.post("I only have xetex")

In [ ]:
show_messages(client.messages)

## Summary

We have seen how to improve LLM answers by retrieving facts from an external source and have the LLM consider those fact in its answers.

While this is an important technique in its own right, it will truly shine if implemented as a skill, see tutorial 'LLM-Agent-skills'. 